# LBM D2Q9 — GPU build, tests and benchmarks on Google Colab

Runs the CUDA versions on a Colab GPU (Tesla T4, `sm_75`):
build in float and double, run all tests, check the kernels with `compute-sanitizer`,
measure the GPU memory bandwidth and benchmark the solver. The CSV files produced in
`results/` are then downloaded and committed to the repository.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

## 1. Environment (GPU, driver, CUDA toolkit)

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version | head -1
!nvidia-smi --query-gpu=name,driver_version,ecc.mode.current,clocks.max.memory --format=csv

## 2. Get the code

In [ ]:
REPO = "https://github.com/H-HABCHANE/Simulation-de-fluide-par-la-m-thode-de-Boltzmann-sur-r-seau-LBM-du-CPU-au-multi-GPU.git"
!rm -rf lbm && git clone --quiet {REPO} lbm
%cd lbm
!git log --oneline -1

## 3. Build (float and double) and run all tests

In [ ]:
for prec in ["OFF", "ON"]:
    d = f"build-double" if prec == "ON" else "build-float"
    !cmake -S . -B {d} -DLBM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLBM_REAL_DOUBLE={prec} > /dev/null
    !cmake --build {d} -j 4 2>&1 | grep -Ei "warning|error" || true
    !ctest --test-dir {d} --output-on-failure

## 4. `compute-sanitizer`

Each tool must report `ERROR SUMMARY: 0 errors`.
- `memcheck`: out-of-bounds and misaligned accesses;
- `initcheck`: reads of uninitialised device memory;
- `racecheck`: shared-memory races (v2 uses none);
- `synccheck`: invalid synchronisation.

In [ ]:
for tool in ["memcheck", "initcheck", "racecheck", "synccheck"]:
    print(f"===== {tool} =====")
    !compute-sanitizer --tool {tool} --error-exitcode 1 build-float/tests/test_v2_vs_v0 2>&1 | tail -3

## 5. GPU memory bandwidth and MLUPS ceiling

In [ ]:
!mkdir -p results
!build-float/gpu_bandwidth results/gpu_bandwidth_Tesla-T4.csv
!cat results/gpu_bandwidth_Tesla-T4.csv
# 320 GB/s: Tesla T4 datasheet value.
!python3 bench/ceiling.py results/gpu_bandwidth_Tesla-T4.csv --gpu --peak-gbs 320

## 6. Benchmarks of v2

1. Block size sweep on one large grid.
2. All grid sizes with the default block (32x8).
3. Same with FMA contraction allowed (`LBM_CUDA_FMAD=ON`), to measure its effect only.

In [ ]:
SIZES = "128x128 512x512 1024x1024 2048x2048 4096x4096"
!python3 bench/run_bench.py --exe build-float/src/v2_cuda_naive/lbm_v2 --sizes 2048x2048 --blocks 32x8 16x16 64x4 128x2 32x16 --updates 5e8 --label v2_blocks --csv results/v2_blocks_Tesla-T4.csv
!python3 bench/run_bench.py --exe build-float/src/v2_cuda_naive/lbm_v2 --sizes {SIZES} --updates 5e8
!python3 bench/run_bench.py --exe build-double/src/v2_cuda_naive/lbm_v2 --sizes 1024x1024 2048x2048 --updates 2e8 --label v2_double --csv results/v2_double_Tesla-T4.csv

In [ ]:
!cmake -S . -B build-fmad -DLBM_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLBM_CUDA_FMAD=ON > /dev/null && cmake --build build-fmad -j 4 --target lbm_v2 > /dev/null
!python3 bench/run_bench.py --exe build-fmad/src/v2_cuda_naive/lbm_v2 --sizes 2048x2048 4096x4096 --updates 5e8 --label v2_fmad --csv results/v2_fmad_Tesla-T4.csv

## 7. Download the results (to be copied into `results/` and committed)

In [ ]:
!ls -l results/
!cd results && zip -q ../results_colab.zip *.csv
from google.colab import files
files.download("results_colab.zip")